# Tucker / HOOI 基礎実装

このNotebookでは、`00_tucker_hosvd_basics.ipynb` で学んだ **HOSVDを初期値として、HOOI (Higher-Order Orthogonal Iteration) でfactor matrixを反復改善する流れ**を確認する。

HOSVDでは各modeのfactorを **元のテンソル `X` のunfoldingから1回ずつ** 求めた。

HOOIでは、あるmode \(n\) のfactor \(U_n\) を更新するとき、

1. それ以外のmodeを現在のfactorで射影する
2. 得られたテンソルをmode-\(n\) unfoldingする
3. その行列の上位左特異ベクトルで \(U_n\) を更新する

という処理を各modeについて繰り返す。

このNotebookでは **全modeを圧縮する通常のTucker/HOOI** に絞る。partial TuckerやCNNへの適用はここでは扱わない。

> **ゴール**
>
> - HOSVDとHOOIの違いを説明できる
> - 「他modeを固定して1つのfactorを更新する」処理を書ける
> - HOOIの反復で再構成誤差がどう変わるか確認できる
> - 自作HOOIとTensorLyのTucker(HOOI)を再構成誤差で比較できる


## 1. 実験対象テンソルとrank

`01_rank_error_tradeoff.ipynb` と同じ形の3階テンソルを使う。

今回は

```python
X.shape = (6, 5, 4)
ranks = {0: 3, 1: 2, 2: 2}
```

で固定する。

rank探索が目的ではない。**同じrankのまま、HOSVDからHOOIへ進むとfactorがどう改善されるか**を見る。


In [ ]:
import torch

from nn_compression.compression import (
    hosvd,
    reconstruct_tucker,
    truncated_svd,
)
from nn_compression.metrics import relative_frobenius_error
from nn_compression.tensor import mode_dot, unfold

torch.manual_seed(0)

X = torch.randn(6, 5, 4)
ranks = {0: 3, 1: 2, 2: 2}

print("shape:", X.shape)
print("ranks:", ranks)


## 2. HOSVDを初期値にする

まず、既存の `hosvd(X, ranks)` で初期factorを作る。

ここで使う `hosvd` は **HOOIではない**。

各factorは元の `X` のunfoldingから1回だけ求めており、他modeのfactorを使った反復更新はしていない。

HOOIではこのHOSVD結果を初期値として使う。


In [ ]:
core_hosvd, factors_hosvd = hosvd(X, ranks)
X_hat_hosvd = reconstruct_tucker(core_hosvd, factors_hosvd)
hosvd_error = relative_frobenius_error(X, X_hat_hosvd)

print("HOSVD core shape:", tuple(core_hosvd.shape))
for mode, U in factors_hosvd.items():
    print(f"mode={mode}: U.shape={tuple(U.shape)}")
print("HOSVD relative error:", float(hosvd_error))


## 3. HOOIで1つのfactorを更新すると何をしているか

まず **mode 0だけ** を考える。

現在のfactorが

```text
U0: (6, 3)
U1: (5, 2)
U2: (4, 2)
```

なら、`U0` を更新するときは `U0` 自身はいったん使わず、

\[
Y
=
X
\times_1 U_1^\mathsf{T}
\times_2 U_2^\mathsf{T}
\]

を作る。

shapeは

```text
X           : (6, 5, 4)
×1 U1.T     : (6, 2, 4)
×2 U2.T     : (6, 2, 2)
```

となる。

次に `Y` をmode 0でunfoldすると

```text
Y_(0): (6, 4)
```

になる。

この行列の上位 `rank[0] = 3` 本の左特異ベクトルを、新しい `U0` とする。

**ポイント:** 更新対象mode以外を現在のfactorで圧縮してからSVDする。これが、各modeを元の `X` から独立に求めるHOSVDとの違い。


In [ ]:
# mode 0 のfactorを1回だけ更新してみる
mode = 0

# HOSVD factorを壊さないようcopyする
factors_work = {
    m: U.clone()
    for m, U in factors_hosvd.items()
}

# TODO:
# 1. projected = X から始める
# 2. mode 1, 2 に U.T を mode_dot する
# 3. projected を mode 0 で unfold する
# 4. truncated_svd(..., ranks[0]) の U を updated_u0 とする

projected = None
updated_u0 = None

# 実装後のshape確認
# print("projected shape:", tuple(projected.shape))   # 期待: (6, 2, 2)
# print("updated U0 shape:", tuple(updated_u0.shape)) # 期待: (6, 3)


### ここで確認すること

`U0` を更新するときに、

```python
unfold(X, 0)
```

をそのままSVDしてしまうと、それはHOSVDのfactorをもう一度求めているだけになる。

HOOIでは必ず、**他modeを現在のfactorで射影したテンソル**を作ってから、更新対象modeをunfoldする。


## 4. 1 sweep分のHOOIを実装する

1 sweepでは、各modeを1回ずつ更新する。

```text
mode 0 を更新
↓
mode 1 を更新
↓
mode 2 を更新
```

mode 1を更新するときは、すでに更新された新しい `U0` を使ってよい。

つまり、1 sweepの途中でも **その時点で最新のfactor** を使う。


In [ ]:
def hooi_sweep(
    X: torch.Tensor,
    factors: dict[int, torch.Tensor],
    ranks: dict[int, int],
) -> dict[int, torch.Tensor]:
    """
    目的:
        HOOIを1 sweepだけ実行し、各modeのfactorを1回ずつ更新する。

    X:
        Tucker分解する元テンソル。

    factors:
        {mode: factor_matrix}。
        HOSVDなどで初期化済みとする。

    ranks:
        {mode: rank}。
        このNotebookでは全modeが入っている前提。

    返り値:
        更新後のfactor辞書。
    """
    updated_factors = {
        mode: U.clone()
        for mode, U in factors.items()
    }

    for mode, rank in ranks.items():
        # TODO:
        # 1. projected = X から始める
        # 2. mode 以外の各 other_mode について
        #       projected = mode_dot(projected, U_other.T, other_mode)
        # 3. unfold(projected, mode) を truncated_svd
        # 4. 左特異ベクトル U を updated_factors[mode] に保存
        pass

    return updated_factors


## 5. factorからcoreを作り直す

HOOIでfactorを更新した後、coreは

\[
G
=
X
\times_0 U_0^\mathsf{T}
\times_1 U_1^\mathsf{T}
\times_2 U_2^\mathsf{T}
\]

で作り直せる。

`hosvd()` のようにfactorを新しく求める必要はない。**現在のfactorを使って元の `X` を全modeで射影するだけ**。


In [ ]:
def core_from_factors(
    X: torch.Tensor,
    factors: dict[int, torch.Tensor],
) -> torch.Tensor:
    """
    目的:
        現在のfactor matrixからTucker coreを計算する。

    ヒント:
        core = X から始め、
        各modeで U.T を mode_dot する。
    """
    # TODO
    pass


## 6. HOOIを反復する

次に、

```text
HOSVDで初期化
↓
HOOI sweep
↓
coreを作る
↓
再構成
↓
relative Frobenius error
↓
収束していなければ次のsweep
```

を繰り返す。

このNotebookでは、各sweep後のerrorを `history` に残す。

### 収束判定

簡単には、前回と今回の誤差差

\[
|e_{k-1} - e_k|
\]

が `tol` 未満になったら止める。

HOOIは、同じmultilinear rankの中でfactorを反復改善するため、**HOSVD初期値より再構成誤差が小さくなるか**に注目する。


In [ ]:
def hooi(
    X: torch.Tensor,
    ranks: dict[int, int],
    max_iter: int = 20,
    tol: float = 1e-6,
) -> tuple[
    torch.Tensor,
    dict[int, torch.Tensor],
    list[float],
]:
    """
    目的:
        HOSVDを初期値としてHOOIを反復する。

    返り値:
        core:
            最終core tensor。

        factors:
            最終factor matrix辞書。

        history:
            HOSVD初期値と各sweep後のrelative Frobenius error。
    """
    # TODO:
    # 1. hosvd(X, ranks) で初期化
    # 2. HOSVD初期値のerrorをhistoryへ入れる
    # 3. max_iter回まで
    #       factors = hooi_sweep(...)
    #       core = core_from_factors(...)
    #       X_hat = reconstruct_tucker(...)
    #       error = relative_frobenius_error(...)
    #       historyへ追加
    #       前回との差がtol未満ならbreak
    # 4. core, factors, history を返す
    pass


## 7. HOSVDとHOOIの再構成誤差を比較する

`hooi()` を完成させたら、同じ `X`・同じ `ranks` で比較する。

見るのは **factorそのものの一致ではなく、再構成誤差**。

factor matrixは符号反転や基底の取り方が異なることがあるため、値の完全一致を比較する必要はない。


In [ ]:
# hooi() 実装後に実行する
# core_hooi, factors_hooi, history = hooi(
#     X,
#     ranks,
#     max_iter=30,
#     tol=1e-7,
# )
#
# X_hat_hooi = reconstruct_tucker(core_hooi, factors_hooi)
# hooi_error = relative_frobenius_error(X, X_hat_hooi)
#
# print("HOSVD error:", float(hosvd_error))
# print("HOOI  error:", float(hooi_error))
# print("iterations:", len(history) - 1)
# print("history:", history)
#
# 確認したいこと:
# - history[0] は HOSVD error
# - HOOI後のerror <= HOSVD error になっているか
# - 数sweep後に改善量が小さくなっていくか


### 誤差の推移を可視化する

HOOIでは最終値だけでなく、**どのように収束したか**を見ると反復法であることが分かりやすい。


In [ ]:
# import matplotlib.pyplot as plt
#
# plt.plot(range(len(history)), history, marker="o")
# plt.xlabel("Sweep (0 = HOSVD initialization)")
# plt.ylabel("Relative Frobenius error")
# plt.title("HOSVD initialization -> HOOI")
# plt.grid(True)
# plt.show()


## 8. TensorLyのHOOIと比較する

TensorLyの `tensorly.decomposition.tucker` は、`init="svd"` の場合、SVD系の初期値からHOOIでfactorを反復改善する。

自作HOOIと比較するときは、

- core / factorのshape
- 再構成tensorのshape
- relative Frobenius error

を見る。

**factorの数値そのものを完全一致させる必要はない。**


In [ ]:
# TensorLyをインストール済みなら実行
#
# import tensorly as tl
# from tensorly.decomposition import tucker
# from tensorly.tucker_tensor import tucker_to_tensor
#
# tl.set_backend("pytorch")
#
# rank_list = [ranks[mode] for mode in range(X.ndim)]
#
# core_tl, factors_tl = tucker(
#     X,
#     rank=rank_list,
#     init="svd",
#     n_iter_max=100,
#     tol=1e-8,
# )
#
# X_hat_tl = tucker_to_tensor((core_tl, factors_tl))
# tensorly_error = relative_frobenius_error(X, X_hat_tl)
#
# print("self HOSVD :", float(hosvd_error))
# print("self HOOI  :", float(hooi_error))
# print("TensorLy   :", float(tensorly_error))
#
# for mode, U in enumerate(factors_tl):
#     print(f"TensorLy mode={mode}: U.shape={tuple(U.shape)}")


## 9. HOSVDとHOOIの違いを整理する

実装後、自分の言葉で次を説明する。

| 観点 | HOSVD | HOOI |
| --- | --- | --- |
| factorの求め方 | 各modeを元の `X` から1回で求める | 他modeを現在のfactorで射影して反復更新 |
| 反復 | しない | する |
| 初期値 | 不要 | HOSVDを使える |
| 計算量 | 比較的軽い | sweep回数だけ追加計算 |
| 同じrankでの誤差 | 初期近似 | HOSVDより改善できることがある |

### このNotebookの完了条件

1. mode 0の1回更新を自分で書けた
2. `hooi_sweep()` を実装できた
3. `core_from_factors()` を実装できた
4. `hooi()` で誤差履歴を取れた
5. HOSVDからHOOIへ進むと誤差がどう変わるか確認できた
6. TensorLyのHOOIと再構成誤差で比較できた

ここまでできれば、Tucker分解そのものの学習としては **HOSVD → rank trade-off → HOOI** の流れを一通り確認できたことになる。
